In [ ]:
import os
from pathlib import Path

import xarray as xr

In [ ]:
ds = xr.open_dataset(os.path.join(Path.home(), "FjordSim_data", "oslofjord", "forcing_105to232.nc"))

In [ ]:
ds

In [ ]:
ds.NUT.isel(time=0, Nz=17).plot()

## BGC Forcing from CSV Files

Load dissolved organic matter (DOM), nutrients (NUT), dissolved oxygen (O2), and phytoplankton (P) climatological profiles from CSV files, interpolate them onto the model grid, and add them as variables to `ds`.

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

data_dir = Path.home() / "FjordSim_data" / "oslofjord"
southern_edge = 10  # NUT is applied only to Ny[:southern_edge]

# Day-of-year index for each time step in ds
ds_doy = ds.time.dt.dayofyear.values.astype(np.float32)

# Model depth levels as positive values, ordered shallow → deep
nz_positive = np.abs(ds.Nz.values)

### Helper functions

In [ ]:
def read_bgc_csv(name, data_dir):
    """Read a BGC CSV file. Returns (values, depths, day_of_year) as float32 arrays."""
    df = pd.read_csv(data_dir / f"{name}.csv", sep=";", index_col=0)
    depths = np.array([float(c.replace("depth_", "")) for c in df.columns])
    doy = df.index.values.astype(np.float32)
    return df.values.astype(np.float32), depths, doy


def interp_profile(vals, csv_depths, csv_doy, nz_positive, ds_doy):
    """Interpolate a (n_csv_days, n_csv_depths) array to (n_ds_time, n_nz).

    Depth axis: linear interpolation; constant extrapolation beyond CSV range
                (boundary value is held for levels deeper than the deepest CSV depth).
    Time axis:  linear interpolation from CSV day-of-year to ds day-of-year;
                boundary values used for any out-of-range days (e.g. leap day 366).
    """
    n_csv_time, n_nz = vals.shape[0], len(nz_positive)

    # Step 1: interpolate depth axis for each CSV day
    depth_interp = np.empty((n_csv_time, n_nz), dtype=np.float32)
    for t in range(n_csv_time):
        depth_interp[t] = np.interp(nz_positive, csv_depths, vals[t])

    # Step 2: interpolate time axis from csv_doy → ds_doy
    f_time = interp1d(csv_doy, depth_interp, axis=0, bounds_error=False,
                      fill_value=(depth_interp[0], depth_interp[-1]))
    return f_time(ds_doy).astype(np.float32)


def make_array(interp_vals, S_nan_mask):
    """Broadcast (time, Nz) → (time, Nz, Ny, Nx) and apply the land mask from S."""
    return np.where(
        S_nan_mask, np.nan, interp_vals[:, :, np.newaxis, np.newaxis]
    ).astype(np.float32)


def make_da(arr, name, ds):
    """Wrap a numpy array in an xarray DataArray with the ds grid coordinates."""
    return xr.DataArray(
        arr, dims=["time", "Nz", "Ny", "Nx"],
        coords={"time": ds.time, "Nz": ds.Nz, "Ny": ds.Ny, "Nx": ds.Nx},
        name=name,
    )

### Read and interpolate CSV profiles

Each CSV contains daily climatological profiles (rows = day of year, columns = depth levels). They are interpolated first onto the model's `Nz` depth levels, then onto the model's time axis.

In [ ]:
dom_vals, dom_depths, dom_doy = read_bgc_csv("DOM", data_dir)
nut_vals, nut_depths, nut_doy = read_bgc_csv("NUT", data_dir)
o2_vals,  o2_depths,  o2_doy  = read_bgc_csv("O2",  data_dir)
p_vals,   p_depths,   p_doy   = read_bgc_csv("P",   data_dir)

dom_interp = interp_profile(dom_vals, dom_depths, dom_doy, nz_positive, ds_doy)
nut_interp = interp_profile(nut_vals, nut_depths, nut_doy, nz_positive, ds_doy)
o2_interp  = interp_profile(o2_vals,  o2_depths,  o2_doy,  nz_positive, ds_doy)
p_interp   = interp_profile(p_vals,   p_depths,   p_doy,   nz_positive, ds_doy)

### Assign to dataset

Broadcast each profile to the full 4-D grid `(time, Nz, Ny, Nx)`, apply the land mask from `S` (NaN where `S` is NaN), and add the variables to `ds`.

For `NUT`, values are assigned only to the southernmost `southern_edge` Ny layers; the remaining rows are left as `NaN`.

In [ ]:
# Land/dry-cell mask derived from salinity
S_nan_mask = np.isnan(ds.S.values)  # (time, Nz, Ny, Nx)

dom_arr = make_array(dom_interp, S_nan_mask)
o2_arr  = make_array(o2_interp,  S_nan_mask)
p_arr   = make_array(p_interp,   S_nan_mask)

# NUT: only Ny[:southern_edge] gets values; the rest stays NaN
nut_full = make_array(nut_interp, S_nan_mask)
nut_arr = np.full_like(nut_full, np.nan)
nut_arr[:, :, :southern_edge, :] = nut_full[:, :, :southern_edge, :]

ds = ds.assign(
    DOM=make_da(dom_arr, "DOM", ds),
    O2=make_da(o2_arr,  "O2",  ds),
    P=make_da(p_arr,    "P",   ds),
    NUT=make_da(nut_arr, "NUT", ds),
)

In [ ]:
ds

In [ ]:
ds["NUT"].isel(time=0, Nz=17).plot()